# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore and analyze the FAIR² dataset using the `mlcroissant` library. Each entity (record set, field, column) is referenced using its `@id` to ensure unique and reproducible identification.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print high-level metadata
md = dataset.metadata
print(f"Dataset Title: {md.name}\n")
print(f"Description: {md.description}\n")
print(f"License: {md.license}")
print(f"Published: {md.datePublished}")
print(f"Keywords: {md.keywords}")

## 2. Data Overview
Review available record sets, their `@id`s, and their fields. All references use their `@id` for clarity and reproducibility.

In [ ]:
# List all available record sets and their fields (referenced by '@id')
record_sets = list(dataset.record_sets)
print(f"Number of record sets found: {len(record_sets)}\n")

for recset in record_sets:
    print(f"Record Set `@id`: {recset['@id']}")
    print(f"    Name: {recset.get('name', 'N/A')}")
    if 'field' in recset:
        if isinstance(recset['field'], list):
            print("    Fields:")
            for field in recset['field']:
                # Each field can be either a dict or an @id string
                if isinstance(field, dict):
                    print(f"      - {field.get('@id', field)}")
                else:
                    print(f"      - {field}")
        else:
            print(f"    Fields: {recset['field']}")
    print()

## 3. Data Extraction
Load data for each record set into a pandas DataFrame for analysis. Record sets and fields are always referenced using their `@id`.

In [ ]:
dataframes = {}
recset_ids = [recset['@id'] for recset in record_sets]

for recset_id in recset_ids:
    print(f"Loading records for record set {recset_id} ...")
    records = list(dataset.records(record_set=recset_id))
    df = pd.DataFrame(records)
    dataframes[recset_id] = df
    print(f"  Retrieved {len(df)} records.")
    if len(df.columns) > 0:
        print(f"  Sample columns ({len(df.columns)}): {df.columns[:min(len(df.columns), 10)].tolist()}")
    print()
# For demonstration, pick the first non-empty DataFrame as a working example
main_record_set_id = None
for recset_id in recset_ids:
    if len(dataframes[recset_id]) > 0:
        main_record_set_id = recset_id
        break

if main_record_set_id is not None:
    print(f"Using record set: {main_record_set_id}")
    df = dataframes[main_record_set_id]
    print(f"Columns in this record set:\n{df.columns.tolist()}")
    display(df.head())
else:
    print("No record sets with data found.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering records based on specific criteria, normalizing numeric fields, and grouping records. All fields are referenced using their `@id`.

In [ ]:
# Automatically select a numeric field by examining dtypes
if main_record_set_id is not None and len(dataframes[main_record_set_id]) > 0:
    df = dataframes[main_record_set_id]
    numeric_cols = df.select_dtypes(include=["number"]).columns.tolist()
    if len(numeric_cols) == 0:
        print("No numeric columns found; EDA limited.")
    else:
        numeric_field_id = numeric_cols[0]
        print(f"Using numeric field: {numeric_field_id}")
        threshold = df[numeric_field_id].median() # Use median as meaningful threshold
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records where {numeric_field_id} > {threshold} (median): {len(filtered_df)} records.")
        display(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / (filtered_df[numeric_field_id].std())
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try to group by a plausible field (first string/object type that's not the numeric id)
        group_field = None
        for c in df.columns:
            if df[c].dtype == object and c != numeric_field_id:
                group_field = c
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(f"Grouped mean {numeric_field_id} by {group_field}:")
            display(grouped_df.head())
else:
    print("No available DataFrame for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. All visualizations use fields referenced by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id is not None and len(dataframes[main_record_set_id]) > 0 and 'numeric_field_id' in locals():
    # Histogram of selected numeric field
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # Boxplot by group_field (if exists)
    if 'group_field' in locals() and group_field:
        # Limit to most common categories to keep the plot readable
        top_groups = df[group_field].value_counts().head(8).index.tolist()
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field, y=numeric_field_id, data=df[df[group_field].isin(top_groups)])
        plt.title(f"{numeric_field_id} grouped by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Visualization unavailable: no suitable numeric or grouping fields.")

## 6. Conclusion
This notebook demonstrated exploration of the FAIR² dataset using the `mlcroissant` library following the Croissant schema. We loaded metadata and data records, reviewed record sets and fields via their `@id`, performed basic exploratory analysis such as filtering and normalization, and visualized key distributions.

For detailed, reproducible scientific data analysis, always reference entities (record sets, fields, columns) by their `@id`. The mlcroissant library enables robust, FAIR (Findable, Accessible, Interoperable, Reusable) data science workflows.